# 09. Leave-One-Fruit-Out (LOFO) Generalization

## 0. Setup

In [1]:
import csv
import random
import re
import json
import time
import math
import copy
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

MANIFEST = Path('~/Desktop/convscript/Code/Manifest/manifests/master_manifest.csv').expanduser()
ROOT     = Path('~/Desktop/Thesis/TR-6').expanduser()
GAS_NORM = Path('~/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json').expanduser()
RUNS     = Path('~/Desktop/convscript/runs').expanduser()

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

LOFO_OUT = RUNS / "09_lofo_generalization"
LOFO_OUT.mkdir(parents=True, exist_ok=True)


Device: mps


## Data pipeline

In [2]:
FRUIT_LIST   = ['Banana', 'Carrot', 'Guava', 'Indian_Gooseberry', 'Mango', 'Tomato']
FRUIT_TO_IDX = {f: i for i, f in enumerate(FRUIT_LIST)}
SESSION_TO_IDX = {'morning': 0, 'afternoon': 1, 'evening': 2}
LABEL_TO_IDX   = {'not_spoiled': 0, 'spoiled': 1}
IMAGE_SIZE     = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
SESSION_HOUR_BUCKETS = {'morning': (5, 11), 'afternoon': (12, 16), 'evening': (16, 19)}
IR_PATTERN   = re.compile(r'(\d{8})_(\d{6})_([\d.]+)C_([\d.]+)C\.jpg$', re.IGNORECASE)
SRGB_PATTERN = re.compile(r'^(\d{8})_(\d{6})[^/]*\.jpg$', re.IGNORECASE)
NUM_FRUITS   = len(FRUIT_TO_IDX)
THRESHOLDS_TO_SWEEP = [t / 100 for t in range(10, 91)]


def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


def hour_to_session(hour):
    for name, (lo, hi) in SESSION_HOUR_BUCKETS.items():
        if lo <= hour < hi:
            return name
    return 'unknown'


def find_ir_folder(base):
    for name in ['IR_fusion_images', 'IR_Fusion_images', 'ir_fusion_images']:
        p = base / name
        if p.exists():
            return p
    return base / 'IR_fusion_images'


def group_images_by_session(folder, pattern):
    result = defaultdict(lambda: defaultdict(list))
    if not folder.exists():
        return {}
    for f in sorted(folder.iterdir()):
        m = pattern.search(f.name)
        if not m:
            continue
        session = hour_to_session(int(m.group(2)[:2]))
        if session != 'unknown':
            result[m.group(1)][session].append(f)
    return dict(result)


def load_image(path, transform):
    return transform(Image.open(path).convert('RGB'))


def get_rgb_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_ir_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

In [3]:
class TrimodalDataset(Dataset):
    def __init__(self, manifest_path, root, train=True, modality_dropout_prob=0.3,
                 exclude_flagged=True, split=None, gas_norm_stats_path=None,
                 cache_images=True, shared_pixel_cache=None, gas_only=False,
                 row_filter=None):
        self.root = Path(root); self.train = train; self.modality_dropout_prob = modality_dropout_prob
        self.rgb_transform = get_rgb_transform(train); self.ir_transform = get_ir_transform(train)
        self.gas_norm_stats = None
        if gas_norm_stats_path:
            with open(gas_norm_stats_path) as f:
                self.gas_norm_stats = json.load(f)['stats']
            print(f'Gas norm stats loaded from {gas_norm_stats_path}')
        with open(manifest_path, newline='') as f:
            all_rows = list(csv.DictReader(f))
        if exclude_flagged:
            all_rows = [r for r in all_rows if r.get('exclude', '').strip().lower() != 'true']
        if split:
            all_rows = [r for r in all_rows if r.get('split', '').strip().lower() == split.lower()]
        if row_filter is not None:
            all_rows = [r for r in all_rows if row_filter(r)]
        self.samples = []; self._build_samples(all_rows)
        self.gas_only = gas_only; self._image_cache = {}
        if not gas_only:
            self._build_image_index()
        self._pixel_cache = {}
        if not gas_only:
            if shared_pixel_cache is not None:
                self._pixel_cache = shared_pixel_cache
                print(f'  Using shared image cache ({len(self._pixel_cache)} images).', flush=True)
            elif cache_images:
                self._warmup_pixel_cache()
        print(f'TrimodalDataset: {len(self.samples)} sessions '
              f'({"train" if train else "val/test"}, split={split}, row_filter={row_filter is not None})')

    def _build_samples(self, rows):
        for row in rows:
            fruit = row['fruit']; label = row['label']; date_str = row['actual_date']
            global_day = int(row['corrected_day_index'])
            days_until = safe_float(row.get('days_until_spoilage', ''), default=-1.0)
            for session in ['morning', 'afternoon', 'evening']:
                si = SESSION_TO_IDX[session]
                ir_avail = safe_float(row.get(f'ir_{session}_count', 0)) > 0
                ir_tmin = safe_float(row.get(f'ir_{session}_tmin', ''), 0.0)
                ir_tmax = safe_float(row.get(f'ir_{session}_tmax', ''), 0.0)
                ir_trange = safe_float(row.get(f'ir_{session}_trange', ''), 0.0)
                srgb_avail = safe_float(row.get(f'srgb_{session}_count', 0)) > 0
                gas_avail = row.get(f'methane_{session}_present', '').strip().upper() == 'TRUE'
                mean_ppm = safe_float(row.get(f'methane_{session}_ppm', ''), 0.0)
                std_ppm = safe_float(row.get(f'methane_{session}_std', ''), 0.0)
                left_ppm = safe_float(row.get(f'methane_{session}_left', ''), 0.0)
                right_ppm = safe_float(row.get(f'methane_{session}_right', ''), 0.0)
                if not ir_avail and not srgb_avail and not gas_avail:
                    continue
                self.samples.append({'fruit': fruit, 'label': label, 'actual_date': date_str,
                    'corrected_day_index': global_day, 'session': session, 'session_idx': si,
                    'fruit_idx': FRUIT_TO_IDX.get(fruit, 0), 'label_idx': LABEL_TO_IDX.get(label, 0),
                    'days_until_spoilage': days_until, 'ir_tmin': ir_tmin, 'ir_tmax': ir_tmax,
                    'ir_trange': ir_trange, 'gas_mean': mean_ppm, 'gas_std': std_ppm,
                    'gas_left': left_ppm, 'gas_right': right_ppm, 'gas_asym': abs(left_ppm - right_ppm),
                    'rgb_available': srgb_avail, 'ir_available': ir_avail, 'gas_available': gas_avail})

    def _build_image_index(self):
        label_map = {'not_spoiled': 'Not_spoiled', 'spoiled': 'Spoiled'}
        for fruit in FRUIT_LIST:
            for lk, lf in label_map.items():
                base = self.root / 'Classified' / fruit / lf
                for d, ss in group_images_by_session(base / 'sRGB_images', SRGB_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['rgb'] = ps
                for d, ss in group_images_by_session(find_ir_folder(base), IR_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['ir'] = ps
        banana_ir = group_images_by_session(find_ir_folder(self.root / 'Normal' / 'Banana'), IR_PATTERN)
        spoiled_dates = {date for (fr, lb, date, _) in self._image_cache if fr == 'Banana' and lb == 'spoiled'}
        for d, ss in banana_ir.items():
            if d in spoiled_dates:
                for s, ps in ss.items():
                    key = ('Banana', 'spoiled', d, s)
                    self._image_cache.setdefault(key, {'rgb': [], 'ir': []})
                    if not self._image_cache[key].get('ir'):
                        self._image_cache[key]['ir'] = ps

    def _load_cached(self, path, transform):
        cached = self._pixel_cache.get(str(path))
        if cached is not None:
            return transform(Image.fromarray(cached.permute(1, 2, 0).numpy()))
        return load_image(path, transform)

    def _warmup_pixel_cache(self):
        all_paths = set()
        for v in self._image_cache.values():
            all_paths.update(v.get('rgb', [])); all_paths.update(v.get('ir', []))
        total = len(all_paths)
        print(f'  Warming up image cache: {total} images...', flush=True)
        for i, path in enumerate(all_paths):
            if str(path) in self._pixel_cache:
                continue
            try:
                img = self.rgb_transform.transforms[0](Image.open(path).convert('RGB'))
                self._pixel_cache[str(path)] = torch.from_numpy(np.array(img)).permute(2, 0, 1)
            except Exception:
                pass
            if (i + 1) % 500 == 0 or (i + 1) == total:
                mb = sum(t.nbytes for t in self._pixel_cache.values()) / (1024 ** 2)
                print(f'  {i + 1}/{total} cached  ({mb:.0f} MB used)', flush=True)
        print(f'  Cache ready. {len(self._pixel_cache)} images in RAM.', flush=True)

    def _normalize_gas(self, fruit, session, mean_ppm, std_ppm, left_ppm, right_ppm):
        if self.gas_norm_stats is None:
            return mean_ppm, std_ppm, left_ppm, right_ppm
        s = self.gas_norm_stats.get(fruit, {}).get(session, {})

        def norm(val, key):
            st = s.get(key, {'mean': 0., 'std': 1.})
            return (val - st['mean']) / st['std']
        return norm(mean_ppm, 'mean_ppm'), norm(std_ppm, 'std_ppm'), norm(left_ppm, 'left_ppm'), norm(right_ppm, 'right_ppm')

    def _apply_dropout(self, rgb_avail, ir_avail, gas_avail):
        if not self.train:
            return rgb_avail, ir_avail, gas_avail
        while True:
            r = rgb_avail and (random.random() > self.modality_dropout_prob)
            i = ir_avail and (random.random() > self.modality_dropout_prob)
            g = gas_avail and (random.random() > self.modality_dropout_prob)
            if r or i or g:
                return r, i, g

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fruit = s['fruit']; label = s['label']; date_str = s['actual_date']
        session = s['session']; si = s['session_idx']
        rgb_avail, ir_avail, gas_avail = self._apply_dropout(s['rgb_available'], s['ir_available'], s['gas_available'])
        cached = self._image_cache.get((fruit, label, date_str, session), {'rgb': [], 'ir': []})
        if not self.gas_only and rgb_avail and cached['rgb']:
            rgb_tensors = torch.stack([self._load_cached(p, self.rgb_transform) for p in cached['rgb']])
        else:
            rgb_avail = False; rgb_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if not self.gas_only and ir_avail and cached['ir']:
            ir_tensors = torch.stack([self._load_cached(p, self.ir_transform) for p in cached['ir']])
        else:
            ir_avail = False; ir_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        ir_scalars = torch.tensor([s['ir_tmin'], s['ir_tmax'], s['ir_trange']], dtype=torch.float32) if ir_avail else torch.zeros(3)
        if gas_avail:
            m, sd, l, r = self._normalize_gas(fruit, session, s['gas_mean'], s['gas_std'], s['gas_left'], s['gas_right'])
            gas = torch.tensor([m, sd, l, r, abs(l - r), float(si) / 2.], dtype=torch.float32)
        else:
            gas = torch.zeros(6)
        return {'rgb_images': rgb_tensors, 'ir_images': ir_tensors, 'ir_scalars': ir_scalars, 'gas': gas,
                'rgb_available': torch.tensor(rgb_avail, dtype=torch.bool),
                'ir_available': torch.tensor(ir_avail, dtype=torch.bool),
                'gas_available': torch.tensor(gas_avail, dtype=torch.bool),
                'fruit_idx': torch.tensor(s['fruit_idx'], dtype=torch.long),
                'session_idx': torch.tensor(si, dtype=torch.long),
                'label': torch.tensor(s['label_idx'], dtype=torch.long),
                'days_until_spoilage': torch.tensor(s['days_until_spoilage'], dtype=torch.float32),
                'fruit': fruit, 'actual_date': date_str, 'corrected_day_index': s['corrected_day_index']}


In [4]:
class DayLevelSequenceDataset(Dataset):
    def __init__(self, manifest_path=MANIFEST, root=ROOT, split="train",
                 gas_norm_stats_path=GAS_NORM, shared_pixel_cache=None, row_filter=None):
        base = TrimodalDataset(
            manifest_path, root, train=False, modality_dropout_prob=0.0,
            split=split, gas_norm_stats_path=gas_norm_stats_path, cache_images=True,
            shared_pixel_cache=shared_pixel_cache, row_filter=row_filter,
        )
        self._pixel_cache = base._pixel_cache
        print(f"DayLevelSequenceDataset (split={split}, row_filter={row_filter is not None}): "
              f"wrapping {len(base)} sessions")
        day_groups = defaultdict(list)
        for i in range(len(base)):
            item = base[i]
            key = (item["fruit"], int(item["label"].item()), int(item["corrected_day_index"]))
            day_groups[key].append(item)
        day_entries = {}
        for (fruit, label, day_idx), sessions in day_groups.items():
            sessions.sort(key=lambda s: int(s["session_idx"].item()))
            rgb_img, ir_img = None, None
            for s in sessions:
                if rgb_img is None and bool(s["rgb_available"].item()) and len(s["rgb_images"]) > 0:
                    rgb_img = s["rgb_images"][0]
                if ir_img is None and bool(s["ir_available"].item()) and len(s["ir_images"]) > 0:
                    ir_img = s["ir_images"][0]
            gas_vals = [s["gas"] for s in sessions if bool(s["gas_available"].item())]
            gas_avail = len(gas_vals) > 0
            gas_vec = torch.stack(gas_vals).mean(dim=0) if gas_avail else torch.zeros(6)
            day_entries.setdefault((fruit, label), []).append({
                "day_idx": day_idx, "rgb_img": rgb_img, "ir_img": ir_img,
                "gas_vec": gas_vec, "gas_avail": gas_avail,
                "fruit_idx": sessions[0]["fruit_idx"], "label": sessions[0]["label"],
                "days_until": sessions[0]["days_until_spoilage"],
            })
        self.trajectories = []
        for (fruit, label), days in day_entries.items():
            days.sort(key=lambda d: d["day_idx"])
            last_observed_idx = None
            for d in days:
                d["delta"] = 0.0 if last_observed_idx is None else float(d["day_idx"] - last_observed_idx)
                if d["gas_avail"]:
                    last_observed_idx = d["day_idx"]
            self.trajectories.append({"fruit": fruit, "label": label, "days": days})
        n_days_total = sum(len(t["days"]) for t in self.trajectories)
        print(f"  {len(self.trajectories)} trajectories, {n_days_total} total day-entries")

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        traj = self.trajectories[idx]["days"]
        T = len(traj)
        blank_rgb = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        blank_ir = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        rgb_seq = torch.stack([d["rgb_img"] if d["rgb_img"] is not None else blank_rgb for d in traj])
        ir_seq = torch.stack([d["ir_img"] if d["ir_img"] is not None else blank_ir for d in traj])
        rgb_avail = torch.tensor([d["rgb_img"] is not None for d in traj], dtype=torch.bool)
        ir_avail = torch.tensor([d["ir_img"] is not None for d in traj], dtype=torch.bool)
        gas_seq = torch.stack([d["gas_vec"] for d in traj])
        gas_mask = torch.tensor([d["gas_avail"] for d in traj], dtype=torch.float32)
        gas_delta = torch.tensor([d["delta"] for d in traj], dtype=torch.float32)
        return {
            "rgb_seq": rgb_seq, "ir_seq": ir_seq, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas_seq, "gas_mask": gas_mask, "gas_delta": gas_delta, "T": T,
            "fruit_idx": traj[0]["fruit_idx"], "label": traj[-1]["label"],
            "days_until": traj[-1]["days_until"], "fruit": self.trajectories[idx]["fruit"],
        }


def day_sequence_collate(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)
    rgb = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    ir = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    rgb_avail = torch.zeros(B, max_T, dtype=torch.bool)
    ir_avail = torch.zeros(B, max_T, dtype=torch.bool)
    gas = torch.zeros(B, max_T, 6)
    gas_mask = torch.zeros(B, max_T)
    gas_delta = torch.zeros(B, max_T)
    pad_mask = torch.ones(B, max_T, dtype=torch.bool)
    fruit_idx = torch.zeros(B, dtype=torch.long)
    label = torch.zeros(B, dtype=torch.long)
    days_until = torch.zeros(B, dtype=torch.float32)
    fruits = []
    for i, b in enumerate(batch):
        T = b["T"]
        rgb[i, :T] = b["rgb_seq"]; ir[i, :T] = b["ir_seq"]
        rgb_avail[i, :T] = b["rgb_avail"]; ir_avail[i, :T] = b["ir_avail"]
        gas[i, :T] = b["gas_seq"]; gas_mask[i, :T] = b["gas_mask"]; gas_delta[i, :T] = b["gas_delta"]
        pad_mask[i, :T] = False
        fruit_idx[i] = b["fruit_idx"]; label[i] = b["label"]; days_until[i] = b["days_until"]
        fruits.append(b.get("fruit"))
    return {"rgb_seq": rgb, "ir_seq": ir, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas, "gas_mask": gas_mask, "gas_delta": gas_delta, "pad_mask": pad_mask,
            "fruit_idx": fruit_idx, "label": label, "days_until": days_until, "fruit": fruits}


class OfflineAugmentedDayLevelSequenceDataset(Dataset):
    def __init__(self, base_dataset, transform_type="flip_rotation", n_copies=1):
        assert transform_type == "flip_rotation"
        self.base = base_dataset
        self.transform_type = transform_type
        self.n_copies = n_copies

        self.index_map = []
        for traj_idx in range(len(base_dataset)):
            self.index_map.append((traj_idx, 0))
            for copy_id in range(1, n_copies + 1):
                self.index_map.append((traj_idx, copy_id))

        self.trajectories = [base_dataset.trajectories[traj_idx] for traj_idx, _ in self.index_map]

        n_orig = len(base_dataset)
        print(f"OfflineAugmentedDayLevelSequenceDataset ({transform_type}, n_copies={n_copies}): "
              f"{n_orig} original trajectories -> {len(self)} total ({n_copies}x duplication)")

    def __len__(self):
        return len(self.index_map)

    def _augment_image(self, img, seed):
        g = torch.Generator().manual_seed(seed)
        out = torch.flip(img, dims=[-1])
        angle = (torch.rand(1, generator=g).item() * 30.0) - 15.0
        out = TF.rotate(out, angle)
        return out

    def __getitem__(self, idx):
        traj_idx, copy_id = self.index_map[idx]
        item = self.base[traj_idx]
        if copy_id == 0:
            return item

        seed_base = traj_idx * 10_000 + copy_id * 100
        item = dict(item)
        item["rgb_seq"] = torch.stack([
            self._augment_image(item["rgb_seq"][t], seed_base + t)
            for t in range(item["rgb_seq"].shape[0])
        ])
        item["ir_seq"] = torch.stack([
            self._augment_image(item["ir_seq"][t], seed_base + t)
            for t in range(item["ir_seq"].shape[0])
        ])
        return item


## Model

In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))

    def forward(self, x):
        return torch.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = x * self.channel_attn(x)
        sa = self.spatial_attn(x)
        return x * sa, sa


BACKBONE_FEATURE_DIMS = {"efficientnet_b0": 1280}


def _infer_feature_dim(backbone, img_size=IMAGE_SIZE):
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(1, 3, img_size, img_size)
        out = backbone(dummy)
    return out.shape[1]


def build_cnn_backbone(name: str):
    if name == "efficientnet_b0":
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        backbone = base.features
        return backbone, BACKBONE_FEATURE_DIMS[name]
    elif name == "convnext_tiny":
        base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = base.features
        feat_dim = _infer_feature_dim(backbone)
        BACKBONE_FEATURE_DIMS[name] = feat_dim
        return backbone, feat_dim
    else:
        raise ValueError(f"Unknown backbone: {name}")


class VisualEncoderToggle(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", freeze_backbone=True,
                 use_cbam=True, chunk_size=8):
        super().__init__()
        self.use_cbam = use_cbam
        self.backbone_name = backbone_name
        self.chunk_size = chunk_size
        self.backbone, self.feature_dim = build_cnn_backbone(backbone_name)
        if use_cbam:
            self.cbam = CBAM(self.feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if freeze_backbone:
            self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n: int):
        self.freeze_backbone()
        for child in list(self.backbone.children())[-n:]:
            for p in child.parameters():
                p.requires_grad = True

    def forward(self, seq: torch.Tensor):
        B, T, C, H, W = seq.shape
        flat = seq.reshape(B * T, C, H, W)
        feats_list = []
        needs_ckpt = (torch.is_grad_enabled()
                      and any(p.requires_grad for p in self.backbone.parameters()))
        for start in range(0, flat.shape[0], self.chunk_size):
            chunk = flat[start:start + self.chunk_size]
            if needs_ckpt:
                feat_map = checkpoint(self.backbone, chunk, use_reentrant=False)
            else:
                feat_map = self.backbone(chunk)
            if self.use_cbam:
                feat_map, _ = self.cbam(feat_map)
            feats_list.append(self.pool(feat_map).flatten(1))
        feats = torch.cat(feats_list, dim=0).view(B, T, -1)
        return feats


class GRUDCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, x_mean=None):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        if x_mean is None:
            x_mean = [0.0] * input_dim
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))
        self.W_gamma_x = nn.Linear(1, input_dim)
        self.W_gamma_h = nn.Linear(1, hidden_dim)
        self.gru_cell = nn.GRUCell(input_dim * 2, hidden_dim)

    def forward(self, x_t, m_t, delta_t, x_last, h_prev):
        gamma_x = torch.exp(-torch.clamp(self.W_gamma_x(delta_t), min=0.0))
        x_bar = self.x_mean.unsqueeze(0).expand_as(x_t)
        x_hat = m_t * x_t + (1 - m_t) * (gamma_x * x_last + (1 - gamma_x) * x_bar)
        gamma_h = torch.exp(-torch.clamp(self.W_gamma_h(delta_t), min=0.0))
        h_t = self.gru_cell(torch.cat([x_hat, m_t], dim=-1), gamma_h * h_prev)
        return h_t, x_hat


class GasGRUDSequenceEncoder(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, x_mean=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = GRUDCell(input_dim, hidden_dim, x_mean)

    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        B, T, D = gas_seq.shape
        device = gas_seq.device
        h = torch.zeros(B, self.hidden_dim, device=device)
        x_last = torch.zeros(B, D, device=device)
        h_seq = []
        for t in range(T):
            x_t = gas_seq[:, t]
            m_t = gas_mask[:, t].unsqueeze(-1).expand(-1, D)
            delta_t = gas_delta[:, t].unsqueeze(-1)
            valid_t = (~pad_mask[:, t]).float().unsqueeze(-1)
            h_new, x_hat = self.cell(x_t, m_t, delta_t, x_last, h)
            h = valid_t * h_new + (1 - valid_t) * h
            x_last = torch.where(m_t.bool(), x_t, x_last)
            h_seq.append(h)
        return torch.stack(h_seq, dim=1)


class CrossModalFusion(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads,
                                           dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
        avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
        key_padding_mask = ~avail
        fully_missing = key_padding_mask.all(dim=1)
        if fully_missing.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[fully_missing] = False
        attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                      need_weights=True, average_attn_weights=True)
        out = self.norm(tokens + attended)
        return out.mean(dim=1), attn_w


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=512,
                 dropout=0.1, max_len=100):
        super().__init__()
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, z_seq: torch.Tensor, pad_mask: torch.Tensor):
        z_seq = self.pos_enc(z_seq)
        return self.encoder(z_seq, src_key_padding_mask=pad_mask)


def gather_last_valid(H: torch.Tensor, pad_mask: torch.Tensor):
    device = H.device
    lengths = (~pad_mask).sum(dim=1)
    last_idx = (lengths - 1).clamp(min=0)
    B = H.shape[0]
    return H[torch.arange(B, device=device), last_idx], last_idx


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, n_tasks: int = 3):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses: list):
        total = 0.0
        for i, L_i in enumerate(losses):
            precision = torch.exp(-2 * self.log_sigma[i])
            total = total + 0.5 * precision * L_i + self.log_sigma[i]
        return total

    def get_sigmas(self):
        return torch.exp(self.log_sigma).detach().cpu().numpy()


class TrimodalFusionModel(nn.Module):
    def __init__(self, rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
                 d_model=64, gas_hidden=64, num_heads=4, temporal_layers=2,
                 dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True):
        super().__init__()
        self.d_model = d_model
        self.rgb_backbone_name = rgb_backbone_name
        self.ir_backbone_name = ir_backbone_name

        self.rgb_encoder = VisualEncoderToggle(rgb_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.ir_encoder = VisualEncoderToggle(ir_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.gas_encoder = GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden)

        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model),
                                      nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model),
                                       nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        self.fusion = CrossModalFusion(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.temporal = TemporalTransformer(d_model=d_model, nhead=num_heads,
                                             num_layers=temporal_layers, dropout=dropout)

        self.cls_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(),
                                       nn.Dropout(cls_dropout), nn.Linear(128, 2))
        self.reg_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(),
                                       nn.Dropout(cls_dropout), nn.Linear(128, 1), nn.ReLU())
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(),
                                             nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def freeze_visual_backbones(self):
        self.rgb_encoder.freeze_backbone(); self.ir_encoder.freeze_backbone()

    def unfreeze_visual_last_n(self, n: int):
        self.rgb_encoder.unfreeze_last_n_layers(n)
        self.ir_encoder.unfreeze_last_n_layers(n)

    def unfreeze_visual_full(self):
        self.rgb_encoder.unfreeze_backbone()
        self.ir_encoder.unfreeze_backbone()

    def forward(self, batch: dict):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape

        rgb_feat = self.rgb_encoder(rgb_seq)
        ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)

        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)

        rgb_flat = rgb_proj.reshape(B * T, -1)
        ir_flat = ir_proj.reshape(B * T, -1)
        gas_flat = gas_proj.reshape(B * T, -1)
        rgb_av_flat = rgb_avail.reshape(B * T)
        ir_av_flat = ir_avail.reshape(B * T)
        gas_av_flat = gas_mask.reshape(B * T).bool()
        z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
        z_seq = z_flat.reshape(B, T, -1)

        H = self.temporal(z_seq, pad_mask)
        H_T, last_idx = gather_last_valid(H, pad_mask)
        cls_logits = self.cls_head(H_T)
        reg_output = self.reg_head(H_T)

        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))

        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}

## Training / evaluation helpers

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_weighted_sampler(dataset):
    labels = [int(t["label"]) for t in dataset.trajectories]
    n_total = len(labels); n_spoiled = max(sum(labels), 1); n_fresh = max(n_total - sum(labels), 1)
    w_spoiled = n_total / (2 * n_spoiled); w_fresh = n_total / (2 * n_fresh)
    weights = [w_spoiled if l == 1 else w_fresh for l in labels]
    return WeightedRandomSampler(weights=weights, num_samples=n_total, replacement=True)


def calibrate_threshold(scores, labels, thresholds=None):
    if thresholds is None:
        thresholds = THRESHOLDS_TO_SWEEP
    if len(set(labels)) < 2:
        return 0.5, 0.0
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="binary", zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def compute_loss(out, batch, uncertainty_loss, cfg, device):
    labels = batch["label"].to(device)
    days = batch["days_until"].to(device)
    class_weights = torch.tensor(cfg["cls_class_weights"], device=device)

    cls_loss = nn.CrossEntropyLoss(weight=class_weights)(out["cls_logits"], labels)
    reg_loss = nn.SmoothL1Loss()(out["reg_output"].squeeze(1), days)

    pad_mask = batch["pad_mask"].to(device)
    gas_mask = batch["gas_mask"].to(device)
    gas_seq = batch["gas_seq"].to(device)
    last_idx = out["last_idx"]
    B = labels.shape[0]
    gas_avail_last = gas_mask[torch.arange(B, device=device), last_idx].bool()
    gas_true_last = gas_seq[torch.arange(B, device=device), last_idx]

    if gas_avail_last.any():
        recon_loss = nn.MSELoss()(out["gas_recon"][gas_avail_last], gas_true_last[gas_avail_last])
    else:
        recon_loss = torch.zeros((), device=device)

    total_loss = uncertainty_loss([cls_loss, reg_loss, recon_loss])
    return total_loss, {"cls_loss": cls_loss.item(), "reg_loss": reg_loss.item(), "recon_loss": recon_loss.item()}


def collect_probs_labels(model, loader, cfg):
    model.eval()
    device = cfg["device"]
    all_labels, all_probs, all_fruits = [], [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch)
            probs = torch.softmax(out["cls_logits"], dim=1).cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
            all_fruits.extend(batch.get("fruit", [None] * len(batch["label"])))
    return np.array(all_probs), all_labels, all_fruits


def get_calibrated_threshold(model, calib_loader, cfg):
    probs, labels, _ = collect_probs_labels(model, calib_loader, cfg)
    threshold, _ = calibrate_threshold(probs[:, 1].tolist(), labels)
    return threshold


def run_train_epoch(model, uncertainty_loss, loader, optimizer, cfg):
    model.train(); uncertainty_loss.train()
    device = cfg["device"]
    total_loss = 0.0
    all_labels, all_probs = [], []
    for batch in loader:
        out = model(batch)
        loss, parts = compute_loss(out, batch, uncertainty_loss, cfg, device)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(list(model.parameters()) + list(uncertainty_loss.parameters()), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        probs = torch.softmax(out["cls_logits"], dim=1).detach().cpu().numpy()
        all_labels.extend(batch["label"].cpu().numpy().tolist())
        all_probs.extend(probs.tolist())
    preds = np.array(all_probs).argmax(axis=1).tolist()
    f1 = f1_score(all_labels, preds, average="binary", zero_division=0)
    return {"loss": round(total_loss / max(len(loader), 1), 4), "f1": round(f1, 4)}


def get_cfg():
    return {
        "epochs": 30, "batch_size": 4, "lr": 1e-4, "lr_finetune": 1e-5, "weight_decay": 1e-4,
        "cls_class_weights": [1.0, 2.3], "device": DEVICE,
    }


## Fold definitions

In [7]:
def make_lofo_train_filter(held_out_fruit):
    def _filter(row):
        return row['fruit'] != held_out_fruit and row.get('split', '').strip().lower() == 'train'
    return _filter


def make_lofo_holdout_filter(held_out_fruit):
    def _filter(row):
        return row['fruit'] == held_out_fruit
    return _filter


FOLDS = FRUIT_LIST 
SEEDS = list(range(10)) 

print(f"{len(FOLDS)} folds x {len(SEEDS)} seeds = {len(FOLDS) * len(SEEDS)} training runs")


6 folds x 10 seeds = 60 training runs


In [8]:
FOLDS_TO_RUN = FOLDS      
SEEDS_TO_RUN = SEEDS  

print(f"Configured to run: {len(FOLDS_TO_RUN)} folds x {len(SEEDS_TO_RUN)} seeds "
      f"= {len(FOLDS_TO_RUN) * len(SEEDS_TO_RUN)} training runs")


Configured to run: 6 folds x 10 seeds = 60 training runs


## Data

In [9]:
cfg = get_cfg()
D_MODEL, DROPOUT = 64, 0.5
AUGMENTATION_TRANSFORM = "flip_rotation"
N_COPIES = 1

SHARED_PIXEL_CACHE = {}

## Train

In [10]:
def train_and_evaluate_fold_seed(held_out_fruit, seed, train_loader, train_loader_calib, holdout_loader):
    set_seed(seed)
    ckpt_dir = LOFO_OUT / held_out_fruit / f"seed_{seed}"
    ckpt_path = ckpt_dir / "model_state.pt"

    model = TrimodalFusionModel(
        rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
        d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT, freeze_visual_backbone=True,
    ).to(cfg["device"])

    if ckpt_path.exists():
        print(f"  [{held_out_fruit}] seed {seed}: found existing checkpoint, reloading "
              f"(skip training).")
        model.load_state_dict(torch.load(ckpt_path, map_location=cfg["device"]))
    else:
        uncertainty_loss = UncertaintyWeightedLoss(n_tasks=3).to(cfg["device"])
        visual_backbone_ids = {id(p) for p in list(model.rgb_encoder.backbone.parameters()) +
                                              list(model.ir_encoder.backbone.parameters())}
        other_params = [p for p in model.parameters() if id(p) not in visual_backbone_ids]
        optimizer = AdamW([
            {"params": other_params, "lr": cfg["lr"]},
            {"params": list(model.rgb_encoder.backbone.parameters()) + list(model.ir_encoder.backbone.parameters()),
             "lr": cfg["lr_finetune"]},
            {"params": uncertainty_loss.parameters(), "lr": cfg["lr"]},
        ], weight_decay=cfg["weight_decay"])
        scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"])

        print(f"{'-' * 90}")
        print(f"[FOLD: held out = {held_out_fruit}] seed {seed} -- {cfg['epochs']} epochs, "
              f"trained on {len(train_loader.dataset)} trajectories (incl. augmentation) "
              f"from the other 5 fruits' train split")
        print(f"{'-' * 90}")

        run_start = time.time()
        for epoch in range(1, cfg["epochs"] + 1):
            epoch_start = time.time()
            unfreeze_note = ""
            if epoch == 6:
                model.unfreeze_visual_last_n(1); unfreeze_note = "  [unfreeze last-1]"
            elif epoch == 8:
                model.unfreeze_visual_last_n(2); unfreeze_note = "  [unfreeze last-2]"
            elif epoch == 12:
                model.unfreeze_visual_full(); unfreeze_note = "  [unfreeze full]"

            train_m = run_train_epoch(model, uncertainty_loss, train_loader, optimizer, cfg)
            scheduler.step()
            epoch_time = time.time() - epoch_start

            print(f"  [{held_out_fruit}] seed {seed} | epoch {epoch:2d}/{cfg['epochs']} | "
                  f"train_loss={train_m['loss']:.4f} train_F1={train_m['f1']:.3f} | "
                  f"{epoch_time:.1f}s{unfreeze_note}")

        total_time = time.time() - run_start
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), ckpt_path)
        print(f"  [{held_out_fruit}] seed {seed}: DONE in {total_time / 60:.1f} min, "
              f"final-epoch state saved to {ckpt_path}")
        
    model.eval()
    threshold = get_calibrated_threshold(model, train_loader_calib, cfg)

    probs, labels, fruits = collect_probs_labels(model, holdout_loader, cfg)
    preds = (probs[:, 1] >= threshold).astype(int).tolist()
    correct = [int(p == l) for p, l in zip(preds, labels)]

    return {
        "held_out_fruit": held_out_fruit, "seed": seed, "threshold": round(threshold, 4),
        "n_holdout_trajectories": len(labels), "labels": labels, "preds": preds,
        "probs_spoiled": probs[:, 1].tolist(), "correct": correct,
        "n_correct": sum(correct), "both_correct": sum(correct) == len(labels),
    }

In [ ]:
fold_seed_results = []
FOLD_FAILURES = []
loop_start = time.time()

for held_out_fruit in FOLDS_TO_RUN:
    print(f"\n\nFOLD: held-out fruit = {held_out_fruit}\n")

    fold_train_base = DayLevelSequenceDataset(
        split=None, row_filter=make_lofo_train_filter(held_out_fruit),
        shared_pixel_cache=SHARED_PIXEL_CACHE,
    )
    holdout_ds = DayLevelSequenceDataset(
        split=None, row_filter=make_lofo_holdout_filter(held_out_fruit),
        shared_pixel_cache=SHARED_PIXEL_CACHE,
    )
    print(f"  Held-out '{held_out_fruit}': {len(holdout_ds)} trajectories "
          f"(expected 2: one spoiled, one not_spoiled)")
    if len(holdout_ds) != 2:
        print(f"  [WARNING] Expected exactly 2 held-out trajectories for {held_out_fruit}, "
              f"found {len(holdout_ds)}. Proceeding, but check the manifest for this fruit.")

    aug_ds = OfflineAugmentedDayLevelSequenceDataset(fold_train_base, AUGMENTATION_TRANSFORM, N_COPIES)
    train_loader = DataLoader(aug_ds, batch_size=cfg["batch_size"], sampler=build_weighted_sampler(aug_ds),
                               collate_fn=day_sequence_collate, num_workers=0)
    train_loader_calib = DataLoader(fold_train_base, batch_size=cfg["batch_size"], shuffle=False,
                                     collate_fn=day_sequence_collate, num_workers=0)
    holdout_loader = DataLoader(holdout_ds, batch_size=cfg["batch_size"], shuffle=False,
                                 collate_fn=day_sequence_collate, num_workers=0)

    for seed in SEEDS_TO_RUN:
        try:
            res = train_and_evaluate_fold_seed(held_out_fruit, seed, train_loader,
                                                train_loader_calib, holdout_loader)
            fold_seed_results.append(res)
            print(f"  [{held_out_fruit}] seed {seed}: RESULT -- {res['n_correct']}/"
                  f"{res['n_holdout_trajectories']} held-out trajectories correct "
                  f"(threshold={res['threshold']})")
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] [{held_out_fruit}] seed {seed}: {type(e).__name__}: {e}\n{tb}")
            FOLD_FAILURES.append({"held_out_fruit": held_out_fruit, "seed": seed,
                                   "type": type(e).__name__, "message": str(e), "traceback": tb})

    elapsed = time.time() - loop_start
    print(f"  Fold '{held_out_fruit}' complete. {elapsed / 60:.1f} min elapsed total.")

print(f"\n{len(fold_seed_results)} successful (fold, seed) runs, {len(FOLD_FAILURES)} failures.")
results_raw_df = pd.DataFrame(fold_seed_results)
results_raw_df.to_csv(LOFO_OUT / "lofo_raw_results.csv", index=False)




FOLD: held-out fruit = Banana

Gas norm stats loaded from /Users/sjanani2073/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json
  Using shared image cache (0 images).
TrimodalDataset: 203 sessions (val/test, split=None, row_filter=True)
DayLevelSequenceDataset (split=None, row_filter=True): wrapping 203 sessions
  10 trajectories, 70 total day-entries
Gas norm stats loaded from /Users/sjanani2073/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json
  Using shared image cache (0 images).
TrimodalDataset: 21 sessions (val/test, split=None, row_filter=True)
DayLevelSequenceDataset (split=None, row_filter=True): wrapping 21 sessions
  2 trajectories, 7 total day-entries
  Held-out 'Banana': 2 trajectories (expected 2: one spoiled, one not_spoiled)
OfflineAugmentedDayLevelSequenceDataset (flip_rotation, n_copies=1): 10 original trajectories -> 20 total (1x duplication)


/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_30357/1412036668.py:197: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[FOLD: held out = Banana] seed 0 -- 30 epochs, trained on 20 trajectories (incl. augmentation) from the other 5 fruits' train split
------------------------------------------------------------------------------------------
  [Banana] seed 0 | epoch  1/30 | train_loss=3.3112 train_F1=0.308 | 4.2s
  [Banana] seed 0 | epoch  2/30 | train_loss=2.5736 train_F1=0.250 | 3.7s
  [Banana] seed 0 | epoch  3/30 | train_loss=2.1390 train_F1=0.667 | 2.6s
  [Banana] seed 0 | epoch  4/30 | train_loss=1.5091 train_F1=0.636 | 1.8s
  [Banana] seed 0 | epoch  5/30 | train_loss=1.5134 train_F1=0.667 | 2.1s
  [Banana] seed 0 | epoch  6/30 | train_loss=2.9424 train_F1=0.632 | 8.6s  [unfreeze last-1]
  [Banana] seed 0 | epoch  7/30 | train_loss=2.7400 train_F1=0.560 | 9.3s
  [Banana] seed 0 | epoch  8/30 | train_loss=1.8454 train_F1=0.417 | 6.6s  [unfreeze last-2]
  [Banana] seed 0 | epoch  9/30 | train_loss=2.0229 trai

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_30357/1412036668.py:197: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[FOLD: held out = Banana] seed 1 -- 30 epochs, trained on 20 trajectories (incl. augmentation) from the other 5 fruits' train split
------------------------------------------------------------------------------------------
  [Banana] seed 1 | epoch  1/30 | train_loss=2.0827 train_F1=0.545 | 2.9s
  [Banana] seed 1 | epoch  2/30 | train_loss=2.9832 train_F1=0.455 | 3.3s
  [Banana] seed 1 | epoch  3/30 | train_loss=1.8856 train_F1=0.667 | 2.2s
  [Banana] seed 1 | epoch  4/30 | train_loss=1.9536 train_F1=0.500 | 2.8s
  [Banana] seed 1 | epoch  5/30 | train_loss=2.7585 train_F1=0.545 | 4.8s
  [Banana] seed 1 | epoch  6/30 | train_loss=2.8144 train_F1=0.417 | 10.8s  [unfreeze last-1]
  [Banana] seed 1 | epoch  7/30 | train_loss=2.7672 train_F1=0.769 | 13.0s
  [Banana] seed 1 | epoch  8/30 | train_loss=2.1634 train_F1=0.500 | 7.6s  [unfreeze last-2]
  [Banana] seed 1 | epoch  9/30 | train_loss=2.1545 tr

...

------------------------------------------------------------------------------------------
[FOLD: held out = Mango] seed 1 -- 30 epochs, trained on 20 trajectories (incl. augmentation) from the other 5 fruits' train split
------------------------------------------------------------------------------------------
  [Mango] seed 1 | epoch  1/30 | train_loss=2.0101 train_F1=0.560 | 3.4s
  [Mango] seed 1 | epoch  2/30 | train_loss=2.8319 train_F1=0.522 | 3.5s
  [Mango] seed 1 | epoch  3/30 | train_loss=1.7456 train_F1=0.500 | 2.1s
  [Mango] seed 1 | epoch  4/30 | train_loss=1.9768 train_F1=0.643 | 3.0s
  [Mango] seed 1 | epoch  5/30 | train_loss=2.6579 train_F1=0.571 | 5.1s
  [Mango] seed 1 | epoch  6/30 | train_loss=2.6175 train_F1=0.421 | 11.8s  [unfreeze last-1]
  [Mango] seed 1 | epoch  7/30 | train_loss=2.7495 train_F1=0.783 | 14.4s
  [Mango] seed 1 | epoch  8/30 | train_loss=1.9791 train_F1=0.640 | 8.0s  [unfreeze last-2]
  [Mango] seed 1 | epoch  9/30 | train_loss=2.0694 train_F1=0.5

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_30357/1412036668.py:197: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[FOLD: held out = Mango] seed 2 -- 30 epochs, trained on 20 trajectories (incl. augmentation) from the other 5 fruits' train split
------------------------------------------------------------------------------------------
  [Mango] seed 2 | epoch  1/30 | train_loss=1.8259 train_F1=0.421 | 2.6s
  [Mango] seed 2 | epoch  2/30 | train_loss=1.7762 train_F1=0.600 | 3.4s
  [Mango] seed 2 | epoch  3/30 | train_loss=1.9980 train_F1=0.727 | 3.9s
  [Mango] seed 2 | epoch  4/30 | train_loss=1.7599 train_F1=0.444 | 3.1s
  [Mango] seed 2 | epoch  5/30 | train_loss=1.1910 train_F1=0.727 | 1.4s
  [Mango] seed 2 | epoch  6/30 | train_loss=2.1333 train_F1=0.545 | 9.6s  [unfreeze last-1]
  [Mango] seed 2 | epoch  7/30 | train_loss=2.2492 train_F1=0.300 | 8.9s
  [Mango] seed 2 | epoch  8/30 | train_loss=2.7240 train_F1=0.455 | 10.8s  [unfreeze last-2]
  [Mango] seed 2 | epoch  9/30 | train_loss=1.7785 train_F1=0.69

/var/folders/1m/bg8_bp9s1dzgb1v2l_3t0kx80000gn/T/ipykernel_30357/1412036668.py:197: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


------------------------------------------------------------------------------------------
[FOLD: held out = Mango] seed 3 -- 30 epochs, trained on 20 trajectories (incl. augmentation) from the other 5 fruits' train split
------------------------------------------------------------------------------------------
  [Mango] seed 3 | epoch  1/30 | train_loss=1.7689 train_F1=0.000 | 2.2s
  [Mango] seed 3 | epoch  2/30 | train_loss=1.8283 train_F1=0.444 | 3.9s
  [Mango] seed 3 | epoch  3/30 | train_loss=2.1065 train_F1=0.526 | 4.7s
  [Mango] seed 3 | epoch  4/30 | train_loss=2.6895 train_F1=0.600 | 4.7s
  [Mango] seed 3 | epoch  5/30 | train_loss=3.1478 train_F1=0.444 | 4.2s
  [Mango] seed 3 | epoch  6/30 | train_loss=2.7473 train_F1=0.500 | 9.7s  [unfreeze last-1]
  [Mango] seed 3 | epoch  7/30 | train_loss=2.0730 train_F1=0.429 | 8.0s
  [Mango] seed 3 | epoch  8/30 | train_loss=2.2991 train_F1=0.364 | 10.0s  [unfreeze last-2]
  [Mango] seed 3 | epoch  9/30 | train_loss=2.0019 train_F1=0.54

## Results

In [ ]:
def expand_trajectory_rows(fold_seed_results):
    rows = []
    for r in fold_seed_results:
        for i in range(r["n_holdout_trajectories"]):
            rows.append({
                "held_out_fruit": r["held_out_fruit"], "seed": r["seed"],
                "trajectory_idx": i, "label": r["labels"][i], "pred": r["preds"][i],
                "prob_spoiled": r["probs_spoiled"][i], "correct": r["correct"][i],
            })
    return pd.DataFrame(rows)


traj_df = expand_trajectory_rows(fold_seed_results)
traj_df.to_csv(LOFO_OUT / "lofo_trajectory_level.csv", index=False)

per_seed_rows = []
for r in fold_seed_results:
    f1 = f1_score(r["labels"], r["preds"], average="binary", zero_division=0)
    acc = accuracy_score(r["labels"], r["preds"])
    per_seed_rows.append({
        "held_out_fruit": r["held_out_fruit"], "seed": r["seed"],
        "f1": round(f1, 4), "accuracy": round(acc, 4),
        "n_correct": r["n_correct"], "both_correct": r["both_correct"],
        "threshold": r["threshold"],
    })
per_seed_df = pd.DataFrame(per_seed_rows)
per_seed_df.to_csv(LOFO_OUT / "lofo_per_seed_results.csv", index=False)
per_seed_df


### Per-fold summary: seed-level distribution and the seed-ensemble call

In [ ]:
fold_summary_rows = []
for held_out_fruit in FOLDS_TO_RUN:
    fold_results = [r for r in fold_seed_results if r["held_out_fruit"] == held_out_fruit]
    if not fold_results:
        fold_summary_rows.append({"held_out_fruit": held_out_fruit, "n_seeds_ok": 0})
        continue

    n_seeds_ok = len(fold_results)
    seeds_both_correct = sum(1 for r in fold_results if r["both_correct"])
    mean_f1 = np.mean([f1_score(r["labels"], r["preds"], average="binary", zero_division=0) for r in fold_results])
    mean_acc = np.mean([accuracy_score(r["labels"], r["preds"]) for r in fold_results])

    labels = fold_results[0]["labels"]
    probs_matrix = np.array([r["probs_spoiled"] for r in fold_results]) 
    mean_probs = probs_matrix.mean(axis=0)
    median_threshold = float(np.median([r["threshold"] for r in fold_results]))
    ensemble_preds = (mean_probs >= median_threshold).astype(int).tolist()
    ensemble_correct = [int(p == l) for p, l in zip(ensemble_preds, labels)]

    fold_summary_rows.append({
        "held_out_fruit": held_out_fruit, "n_seeds_ok": n_seeds_ok,
        "seeds_both_correct": seeds_both_correct,
        "seeds_both_correct_frac": round(seeds_both_correct / n_seeds_ok, 3),
        "mean_seed_f1": round(float(mean_f1), 4), "mean_seed_accuracy": round(float(mean_acc), 4),
        "ensemble_median_threshold": round(median_threshold, 4),
        "ensemble_n_correct": sum(ensemble_correct),
        "ensemble_both_correct": sum(ensemble_correct) == len(labels),
    })

fold_summary_df = pd.DataFrame(fold_summary_rows)
fold_summary_df.to_csv(LOFO_OUT / "lofo_fold_summary.csv", index=False)
fold_summary_df

### Primary summary: how many of the 6 folds generalize?

In [ ]:
n_folds_both_correct = int(fold_summary_df["ensemble_both_correct"].sum())
n_folds_total = len(fold_summary_df)

print("=" * 90)
print(f"PRIMARY RESULT: seed-ensemble correctly classifies BOTH held-out trajectories in "
      f"{n_folds_both_correct} / {n_folds_total} folds.")
print("=" * 90)
for _, row in fold_summary_df.iterrows():
    status = "BOTH CORRECT" if row.get("ensemble_both_correct") else f"{row.get('ensemble_n_correct', '?')}/2 correct"
    print(f"  {row['held_out_fruit']:<20} {status:<16} "
          f"(seed-level: {row.get('seeds_both_correct', '?')}/{row.get('n_seeds_ok', '?')} seeds got both right, "
          f"mean seed F1={row.get('mean_seed_f1', float('nan'))})")

if FOLD_FAILURES:
    print(f"\n{len(FOLD_FAILURES)} (fold, seed) run(s) failed")


### Plot: per-trajectory outcome by fold

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
fold_order = list(FOLDS_TO_RUN)
for fi, held_out_fruit in enumerate(fold_order):
    sub = traj_df[traj_df["held_out_fruit"] == held_out_fruit]
    for _, row in sub.iterrows():
        jitter = (row["seed"] - 4.5) * 0.015 
        x = fi + jitter + (0.12 if row["label"] == 1 else -0.12)
        color = "#27ae60" if row["correct"] else "#c0392b"
        marker = "^" if row["label"] == 1 else "o"
        ax.scatter([x], [row["seed"]], color=color, marker=marker, s=40, alpha=0.85)

ax.set_xticks(range(len(fold_order)))
ax.set_xticklabels(fold_order, rotation=30, ha="right")
ax.set_ylabel("seed")
ax.set_title("Per-seed, per-trajectory outcome by held-out fold\n"
              "(triangle = spoiled trajectory, circle = not_spoiled; green = correct, red = incorrect)")
plt.tight_layout()
plt.savefig(LOFO_OUT / "fig_per_fold_outcomes.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(fold_summary_df))
ax.bar(x, fold_summary_df["ensemble_n_correct"], color=["#27ae60" if b else "#e67e22"
                                                          for b in fold_summary_df["ensemble_both_correct"]])
ax.set_xticks(x)
ax.set_xticklabels(fold_summary_df["held_out_fruit"], rotation=30, ha="right")
ax.set_ylabel("Seed-ensemble: held-out trajectories correct (of 2)")
ax.set_ylim(0, 2.2)
ax.set_yticks([0, 1, 2])
ax.set_title(f"Per-fold seed-ensemble outcome ({n_folds_both_correct}/{n_folds_total} folds fully correct)")
for xi, row in zip(x, fold_summary_df.itertuples()):
    ax.text(xi, row.ensemble_n_correct + 0.08, str(row.ensemble_n_correct), ha="center")
plt.tight_layout()
plt.savefig(LOFO_OUT / "fig_ensemble_outcome_by_fold.png", dpi=150)
plt.show()


In [ ]:
print("SUMMARY: does the model generalize to a completely unseen fruit type?")
print(f"\n{n_folds_both_correct} of {n_folds_total} folds: seed-ensemble correctly classifies "
      f"both held-out trajectories.")

mean_seed_f1_overall = fold_summary_df["mean_seed_f1"].mean() if "mean_seed_f1" in fold_summary_df else float("nan")
print(f"Mean per-seed F1 across all folds: {mean_seed_f1_overall:.4f}")

worst_folds = fold_summary_df[~fold_summary_df["ensemble_both_correct"].fillna(False)]
if not worst_folds.empty:
    print(f"\nFolds where the seed-ensemble did NOT get both held-out trajectories correct: "
          f"{', '.join(worst_folds['held_out_fruit'].tolist())}")
    print("Suspect")
else:
    print("\nEvery fold's seed-ensemble got both held-out trajectories correct")

print(f"\nFull results: {LOFO_OUT / 'lofo_raw_results.csv'}")
print(f"Trajectory-level: {LOFO_OUT / 'lofo_trajectory_level.csv'}")
print(f"Per-seed: {LOFO_OUT / 'lofo_per_seed_results.csv'}")
print(f"Fold summary: {LOFO_OUT / 'lofo_fold_summary.csv'}")
